# The Hidden Layer: test cases 2 and 3

**IITGN PG Diploma · Business Analytics**

This notebook covers the Python steps of Test Case 2 (the routing AI at Kalpa Retail) and Test Case 3 (expense claims at Kalpa Financial Services). Test Case 1 is solved on paper.

Work top to bottom. Each task cell says what to compute. Write your answers to the written questions in your handout, not here.

Libraries: NumPy, pandas, SciPy and matplotlib only.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import chisquare

---
## Test Case 2: The routing AI at Kalpa Retail

Model of the network:
- 60 dark stores.
- Each store has a **true** on-time rate drawn from N(80, 5). It never changes.
- Each quarter adds fresh **noise** drawn from N(0, 5).
- The observed rate is true rate plus noise. With these settings the quarter-to-quarter correlation is 0.5.

No store receives any tool in the simulation unless a task says so.

### Task 3. How much do the worst 12 stores improve with no tool?

Simulate one network, select the bottom 12 stores by Q1, and record their average Q2 minus their average Q1. Repeat 5,000 times.

Report the average improvement, its standard deviation, and the share of runs with an improvement of at least 6 points.

In [ ]:
rng = np.random.default_rng(42)
runs = 5000
gains = []

for r in range(runs):
    true = rng.normal(80, 5, 60)          # each store's true rate, fixed
    # TODO: draw Q1 as true rate plus noise from N(0, 5)
    # TODO: draw Q2 as true rate plus fresh noise from N(0, 5)
    # TODO: find the 12 lowest stores in Q1 (hint: np.argsort)
    # TODO: append their Q2 average minus their Q1 average to gains
    pass

gains = np.array(gains)
# TODO: print the average, the standard deviation, and the share of runs with gains >= 6

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(gains, bins=40, color="#9AA3AF", edgecolor="white")
plt.axvline(6, color="#C8471C", linewidth=2, label="Observed improvement: 6 points")
plt.title("Improvement of the worst 12 stores with no tool (5,000 runs)")
plt.xlabel("Q2 average minus Q1 average, points")
plt.ylabel("Number of runs")
plt.legend()
plt.show()

### Task 4. Hotelling's reversal

Simulate **one** network. Group the stores into quarters by **Q2**, then compare each group's Q1 and Q2 averages.

What would Hotelling conclude from this table?

In [ ]:
rng = np.random.default_rng(7)
true = rng.normal(80, 5, 60)
q1 = true + rng.normal(0, 5, 60)
q2 = true + rng.normal(0, 5, 60)

stores = pd.DataFrame({"q1": q1, "q2": q2})
# TODO: add a column that groups stores into four quarters by q2 (hint: pd.qcut)
# TODO: show the average q1 and q2 for each group

### Task 6. A fair test of the tool

Take the bottom 24 stores by Q1. Randomly assign 12 to the tool and 12 to no tool. Give the tool a **true effect of +1 point** in Q2. The estimate is the treated Q2 average minus the control Q2 average.

Repeat 5,000 times and report the mean and standard deviation of the estimate.

In [ ]:
rng = np.random.default_rng(42)
estimates = []

for r in range(5000):
    true = rng.normal(80, 5, 60)
    q1 = true + rng.normal(0, 5, 60)
    q2 = true + rng.normal(0, 5, 60)
    # TODO: take the 24 lowest stores by Q1
    # TODO: randomly choose 12 of them as treated (hint: rng.permutation); the rest are control
    # TODO: add the true effect of 1.0 to the treated stores' Q2
    # TODO: append treated Q2 average minus control Q2 average to estimates
    pass

estimates = np.array(estimates)
# TODO: print the mean and standard deviation of the estimates

---
## Test Case 3: Before the AI approves expense claims

Upload `claims.csv` to this notebook's files (in Colab, use the folder icon on the left), then run the next cell.

Columns: `claim_id`, `employee_id`, `category`, `amount` (rupees).

In [ ]:
claims = pd.read_csv("claims.csv")
print(claims.shape)
claims.head()

Benford's expected share for each first digit, and the first digit of every amount, are set up for you.

In [ ]:
digits = np.arange(1, 10)
benford = pd.Series(np.log10(1 + 1 / digits), index=digits)   # expected share of each first digit

# first digit: divide by the power of ten below the amount, keep the integer part
claims["d1"] = (claims["amount"] / 10 ** np.floor(np.log10(claims["amount"]))).astype(int)
benford.round(3)

### Task 1. Which data does Benford's law apply to?

Inspect the categories and the most common amounts. Is any category set by rule rather than chosen freely?

In [ ]:
# TODO: count claims by category
# TODO: list the five most frequent amounts
# TODO: count distinct amounts within each category

### Task 2. Test all claims, then test without the excluded category

For each version, compute the observed share of each first digit, the mean absolute deviation (MAD) from Benford, and the chi-square test.

In [ ]:
# version A: all claims
# TODO: observed share of each first digit (hint: value_counts(normalize=True).reindex(digits, fill_value=0))
# TODO: MAD = average of the absolute gaps between observed and benford
# TODO: chi-square on the counts against benford * number of claims

# version B: remove the category you identified in Task 1, store the rest as x, and repeat
x = claims.copy()   # TODO: filter out the excluded category

In [ ]:
# TODO: plot Benford, all claims and the cleaned claims side by side as a bar chart

### Task 4. MAD for each employee

Use the cleaned claims `x`. Rank employees by MAD and report the median.

In [ ]:
# TODO: count each employee's claims by first digit (hint: pivot_table with aggfunc="count")
# TODO: turn counts into shares by dividing each row by its total
# TODO: MAD per employee = mean absolute gap from benford across the nine digits
# TODO: sort descending, print the median, and count employees above 0.015

In [ ]:
# TODO: show the first-digit shares (%) of your top three employees

### Task 5. Claims just below and just above the ₹25,000 limit

In [ ]:
# TODO: flag claims between 20,000 and 24,999.99, and between 25,000 and 29,999.99
# TODO: count both flags per employee
# TODO: report the company-wide ratio of below to above, and the employees with the most claims below

### Task 6. Your memo

Write your audit memo in the handout: who you would refer, the evidence for each, one likely false positive, and two controls the AI agent needs.